In [ ]:
# CREATE CSV

from pathlib import Path

import pandas as pd
from astropy.io import fits
from data_types import Header_Content, S4ACS_Config
from utils import create_csv

config = S4ACS_Config.from_config_file("acs_config.cfg")
nights_folder = config.image_path / ".."

# nights = [
#     night.stem
#     for night in nights_folder.iterdir()
#     if "2026" in night.stem and "test" not in night.stem and "temp" not in night.stem
# ]
nights = [
    "20260731",
    "20260702",
    "20260822",
    "20260719",
    "20260805",
    "20260714",
    "20260823",
    "20260819",
    "20260715",
    "20260803",
]
for night in nights:
    csv_file = Path.cwd() / "_csv" / f"{night}.csv"
    create_csv(csv_file)
    config = S4ACS_Config.from_config_file("acs_config.cfg")
    images_folder = config.image_path / ".." / night
    images_list = [
        name.resolve()
        for name in images_folder.iterdir()
        if name.suffix == ".fits" and "test" not in name.name
    ]

    for image_file in images_list:
        hdr = fits.getheader(image_file)
        hdr_cnt = Header_Content.from_header(hdr)
        df = pd.DataFrame([hdr_cnt.to_csv_format])
        df.to_csv(csv_file, mode="a", index=False, header=False)

In [12]:
# FILTER CSVs

import pandas as pd
from pathlib import Path

headers = ["Date", "N. images", "Cycles"]
report = {k: [] for k in headers}

csv_folder = Path.cwd() / "_ch1_csv"
obs_nights = [night.stem for night in csv_folder.iterdir()]

for night in obs_nights:
    report["Date"].append(night)
    df = pd.read_csv(csv_folder / f"{night}.csv")

    df = df[df["INSTMODE"] == "POLAR"]
    df = df[df["OBSTYPE"] == "OBJECT"]
    report["N. images"].append(df.shape[0])
    report["Cycles"].append(df["CYCLE_IDX"].max())

report = pd.DataFrame.from_dict(report)
report = report.sort_values("Cycles", ascending=False)
report_file = Path.cwd() / "report.csv"
report.to_csv(report_file, index=False)
print(report.head(10))
print(report["Date"].head(10).values)


         Date  N. images  Cycles
134  20261116       3071   149.0
47   20260727       1781   100.0
72   20260822       8493    99.0
51   20260731      10306    85.0
38   20260714       4275    76.0
37   20260713       2128    76.0
43   20260719       7231    76.0
73   20260823       3456    70.0
83   20260903       1371    68.0
3    20260604       1984    56.0
<StringArray>
['20261116', '20260727', '20260822', '20260731', '20260714', '20260713',
 '20260719', '20260823', '20260903', '20260604']
Length: 10, dtype: str


In [1]:
# CREATE REPORT

import pandas as pd
from pathlib import Path

headers = ["Date", "N. images", "Cycles"]
report = {k: [] for k in headers}

csv_folder = Path.cwd() / "_ch1_csv"
obs_nights = [
    "20261116",
    "20260727",
    "20260822",
    "20260731",
    "20260714",
    "20260713",
    "20260719",
    "20260823",
    "20260903",
    "20260604",
]
for night in obs_nights:
    report["Date"].append(night)
    df = pd.read_csv(csv_folder / f"{night}.csv")

    df = df[df["INSTMODE"] == "POLAR"]
    df = df[df["OBSTYPE"] == "OBJECT"]
    report["N. images"].append(df.shape[0])
    report["Cycles"].append(int(df["CYCLE_IDX"].max()))

report = pd.DataFrame.from_dict(report)
report = report.sort_values("Date", ascending=True)
# report["Date"] = [f"{date[:4]}--{date[4:6]}--{date[6:]}" for date in report["Date"]]
print(report.head(10).to_latex(index=False))


\begin{tabular}{lrr}
\toprule
Date & N. images & Cycles \\
\midrule
20260604 & 1984 & 56 \\
20260713 & 2128 & 76 \\
20260714 & 4275 & 76 \\
20260719 & 7231 & 76 \\
20260727 & 1781 & 100 \\
20260731 & 10306 & 85 \\
20260822 & 8493 & 99 \\
20260823 & 3456 & 70 \\
20260903 & 1371 & 68 \\
20261116 & 3071 & 149 \\
\bottomrule
\end{tabular}

